# Lab 3: Membership Inference Attack 

**Goal**: Detect if discharge notes were in the training set by analyzing model behavior differences.

**Strategy**: Look for "memorization signatures" - models behave differently on seen vs unseen data.

In [ ]:
# Core imports
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, roc_curve, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set device to use gpu if possible.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Using device: cuda
GPU: NVIDIA A100-SXM4-40GB
GPU Memory: 39.5 GB


In [2]:
# Load dataset and models
print("Loading dataset...")
dataset = load_dataset("UBC-SLIME/colx_531_group_project")

print("Loading models...")
model_135m = AutoModelForCausalLM.from_pretrained("UBC-SLIME/colx_531_smollm2-135m")
model_360m = AutoModelForCausalLM.from_pretrained("UBC-SLIME/colx_531_smollm2-360m")
tokenizer = AutoTokenizer.from_pretrained("UBC-SLIME/colx_531_smollm2-135m")

# Move models to device
model_135m.to(device)
model_360m.to(device)
model_135m.eval()
model_360m.eval()

print(f"Dataset: {len(dataset['train'])} train, {len(dataset['validation'])} validation samples")
print("Models loaded: 135M and 360M parameters")

Loading dataset...


train.json:   0%|          | 0.00/102M [00:00<?, ?B/s]

dev.json:   0%|          | 0.00/20.2M [00:00<?, ?B/s]

test.json:   0%|          | 0.00/30.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/50000 [00:00<?, ? examples/s]

...

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/110 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Dataset: 50000 train, 10000 validation samples
Models loaded: 135M and 360M parameters


## Core Membership Inference Functions

Extract features that indicate memorization patterns from both models.

In [ ]:
def compute_perplexity_and_features(model, tokenizer, text, max_length=512):
    """
    Compute perplexity and other features that might indicate memorization.

    Returns:
        dict with perplexity, loss, confidence metrics.
    """
    # Tokenize text
    inputs = tokenizer(text, return_tensors="pt", max_length=max_length,
                      truncation=True, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        # Get model outputs
        outputs = model(**inputs, labels=inputs["input_ids"])
        logits = outputs.logits

        #Lower loss suggests that the model has seen this pattern before. Higher loss suggests that the model
        #is struggling with never-seen-before patterns
        loss = outputs.loss.item()

        # Calculate perplexity
        #Low perplexity means that the model is more likely to have seen the item before
        #high perplexity more likely means the model hasn't seen it.
        perplexity = torch.exp(outputs.loss).item()

        # Get token-level probabilities
        shift_logits = logits[..., :-1, :]
        shift_labels = inputs["input_ids"][..., 1:]

        # Calculate probabilities for each position
        token_probs = F.softmax(shift_logits, dim=-1)

        #  Extract probability of the actual token that appeared
        # For exach position, what ws the possibility of the word that actually appeared?
        actual_token_probs = token_probs.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1)

        # Token probability statistics
        #average confidence of across all tokens in the text
        mean_token_prob = actual_token_probs.mean().item()

        #The lowest confidence the model had for any single token
        #if this is high, the model is pretty confident even on its lowest confidence
        #can signal that it is more likely that the model has seen this data before
        min_token_prob = actual_token_probs.min().item()

        #highest confidence the model had for any single token. With a very high
        #maximum, means that some words are extremely predictable. 
        max_token_prob = actual_token_probs.max().item()

        #Standard deviation of probabilities per token. If the data has been seen before
        # by the model, the standard deviation will be lower because its confidence will be
        # higher overall. Higher standard deviation would indicate more variability in 
        #the model's confidence level for each token and that would indicate that it is unlikely
        #to have seen the data during training.
        std_token_prob = actual_token_probs.std().item()

        # Entropy / Uncertainty: higher entropy means model is more uncertain.
        #Higher entropy means less likely to have seen data before
        token_entropies = -(token_probs * torch.log(token_probs + 1e-10)).sum(dim=-1)
        mean_entropy = token_entropies.mean().item()

        # Top-k probability mass = how much probability model assigns to its top k most
        #confident predictions. We used k = 10
        #high top k mass = model is confident and focused: likely seen this pattern before
        #low top k mass = model is uncertain and likely hasn't seen this pattern before
        top_k_probs, _ = torch.topk(token_probs, k=10, dim=-1)
        top_k_mass = top_k_probs.sum(dim=-1).mean().item()

        # Low-confidence tokens ratio
        #If the ratio is closer to 0, few tokens had low confidence, which means that the model
        #likely saw this pattern before.
        #higher low-confidence ratio (closer to 1) means model was frequently uncertain and likely
        #hasn't seen this pattern before
        low_conf_tokens = (actual_token_probs < 0.1).sum().item()
        low_conf_ratio = low_conf_tokens / actual_token_probs.shape[1]

    return {
        'perplexity': perplexity,
        'loss': loss,
        'mean_token_prob': mean_token_prob,
        'min_token_prob': min_token_prob,
        'max_token_prob': max_token_prob,
        'std_token_prob': std_token_prob,
        'mean_entropy': mean_entropy,
        'top_k_mass': top_k_mass,
        'low_conf_ratio': low_conf_ratio,
        'num_tokens': actual_token_probs.shape[1]
    }

In [4]:
def analyze_sample_batch(model, tokenizer, texts, labels, model_name="model"):
    """
    Analyze a batch of texts and return features for membership inference.
    """
    results = []

    for text, label in tqdm(zip(texts, labels), total=len(texts),
                           desc=f"Analyzing with {model_name}"):
        try:
            features = compute_perplexity_and_features(model, tokenizer, text)
            features['label'] = label
            features['text_length'] = len(text)
            results.append(features)
        except Exception as e:
            print(f"Error processing text (len={len(text)}): {str(e)[:100]}")
            # Add dummy entry to maintain alignment
            results.append({
                'perplexity': float('inf'),
                'loss': float('inf'),
                'mean_token_prob': 0,
                'label': label,
                'text_length': len(text)
            })

    return pd.DataFrame(results)

In [5]:
def process_dataset_in_batches(model, tokenizer, texts, labels, model_name="model", batch_size=1000):
    """
    Process large dataset in batches for memory efficiency.
    """
    all_results = []

    print(f"Processing {len(texts)} samples in batches of {batch_size}...")

    for i in range(0, len(texts), batch_size):
        end_idx = min(i + batch_size, len(texts))
        batch_texts = texts[i:end_idx]
        batch_labels = labels[i:end_idx]

        print(f"Batch {i//batch_size + 1}/{(len(texts)-1)//batch_size + 1}: {end_idx-i} samples")

        batch_results = analyze_sample_batch(model, tokenizer, batch_texts, batch_labels, model_name)
        all_results.append(batch_results)

        # GPU memory management
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    final_results = pd.concat(all_results, ignore_index=True)
    print(f"Completed processing {len(final_results)} samples with {model_name}")
    return final_results

## Dataset Preparation

Load and prepare the full training and validation datasets.

In [6]:
# Load full dataset splits
train_df = dataset['train'].to_pandas()
val_df = dataset['validation'].to_pandas()
test_df = dataset['test'].to_pandas()

print(f"Training set: {len(train_df)} samples")
print(f"Validation set: {len(val_df)} samples")
print(f"Test set: {len(test_df)} samples")
print(f"Training - Members: {sum(train_df['is_member'])}, Non-members: {len(train_df) - sum(train_df['is_member'])}")
print(f"Validation - Members: {sum(val_df['is_member'])}, Non-members: {len(val_df) - sum(val_df['is_member'])}")

# Extract texts and labels
train_texts = train_df['text'].tolist()
train_labels = train_df['is_member'].tolist()
val_texts = val_df['text'].tolist()
val_labels = val_df['is_member'].tolist()
test_texts = test_df['text'].tolist()
test_labels = [0] * len(test_texts)  # Dummy labels for test set

Training set: 50000 samples
Validation set: 10000 samples
Test set: 15000 samples
Training - Members: 25000, Non-members: 25000
Validation - Members: 5000, Non-members: 5000


## Feature Extraction

Process all training and validation data with both models to extract membership inference features.

In [7]:
# Process training data
print("Processing training data with 135M model...")
train_results_135m = process_dataset_in_batches(
    model_135m, tokenizer, train_texts, train_labels, "135M", batch_size=500
)

print("\nProcessing training data with 360M model...")
train_results_360m = process_dataset_in_batches(
    model_360m, tokenizer, train_texts, train_labels, "360M", batch_size=500
)

print(f"\nTraining data processing complete!")
print(f"135M results: {train_results_135m.shape}")
print(f"360M results: {train_results_360m.shape}")

Processing training data with 135M model...
Processing 50000 samples in batches of 500...
Batch 1/100: 500 samples


Analyzing with 135M: 100%|██████████| 500/500 [00:20<00:00, 24.91it/s]


Batch 2/100: 500 samples


Analyzing with 135M: 100%|██████████| 500/500 [00:18<00:00, 26.38it/s]


Batch 3/100: 500 samples


...

Batch 99/100: 500 samples


Analyzing with 360M: 100%|██████████| 500/500 [00:20<00:00, 24.73it/s]


Batch 100/100: 500 samples


Analyzing with 360M: 100%|██████████| 500/500 [00:20<00:00, 24.89it/s]


Completed processing 50000 samples with 360M

Training data processing complete!
135M results: (50000, 12)
360M results: (50000, 12)


In [8]:
# Process validation data
print("Processing validation data with 135M model...")
val_results_135m = process_dataset_in_batches(
    model_135m, tokenizer, val_texts, val_labels, "135M", batch_size=500
)

print("\nProcessing validation data with 360M model...")
val_results_360m = process_dataset_in_batches(
    model_360m, tokenizer, val_texts, val_labels, "360M", batch_size=500
)

print(f"\nValidation data processing complete!")
print(f"135M results: {val_results_135m.shape}")
print(f"360M results: {val_results_360m.shape}")

Processing validation data with 135M model...
Processing 10000 samples in batches of 500...
Batch 1/20: 500 samples


Analyzing with 135M: 100%|██████████| 500/500 [00:18<00:00, 26.46it/s]


Batch 2/20: 500 samples


Analyzing with 135M: 100%|██████████| 500/500 [00:18<00:00, 26.51it/s]


Batch 3/20: 500 samples


...

Batch 19/20: 500 samples


Analyzing with 360M: 100%|██████████| 500/500 [00:20<00:00, 24.98it/s]


Batch 20/20: 500 samples


Analyzing with 360M: 100%|██████████| 500/500 [00:19<00:00, 25.07it/s]

Completed processing 10000 samples with 360M

Validation data processing complete!
135M results: (10000, 12)
360M results: (10000, 12)


In [9]:
# Process test data
print("Processing test data with 135M model...")
test_results_135m = process_dataset_in_batches(
    model_135m, tokenizer, test_texts, test_labels, "135M", batch_size=500
)

print("\nProcessing test data with 360M model...")
test_results_360m = process_dataset_in_batches(
    model_360m, tokenizer, test_texts, test_labels, "360M", batch_size=500
)

print(f"\nTest data processing complete!")
print(f"135M results: {test_results_135m.shape}")
print(f"360M results: {test_results_360m.shape}")

Processing test data with 135M model...
Processing 15000 samples in batches of 500...
Batch 1/30: 500 samples


Analyzing with 135M: 100%|██████████| 500/500 [00:18<00:00, 26.45it/s]


Batch 2/30: 500 samples


Analyzing with 135M: 100%|██████████| 500/500 [00:18<00:00, 26.55it/s]


Batch 3/30: 500 samples


...

Batch 29/30: 500 samples


Analyzing with 360M: 100%|██████████| 500/500 [00:19<00:00, 25.06it/s]


Batch 30/30: 500 samples


Analyzing with 360M: 100%|██████████| 500/500 [00:19<00:00, 25.13it/s]


Completed processing 15000 samples with 360M

Test data processing complete!
135M results: (15000, 12)
360M results: (15000, 12)


## Membership Inference Classifier

Train classifiers using features from both models and evaluate on the validation set.

In [10]:
def build_membership_classifier(train_135m, train_360m, val_135m, val_360m):
    """
    Build classifier using full training set and evaluate on validation set.
    """
    print("Building membership inference classifier...")

    # Feature columns
    feature_cols = ['perplexity', 'loss', 'mean_token_prob', 'min_token_prob', 'max_token_prob',
                   'std_token_prob', 'mean_entropy', 'top_k_mass', 'low_conf_ratio']

    # Prepare training features
    train_features_135m = train_135m[feature_cols].copy()
    train_features_360m = train_360m[feature_cols].copy()

    # Add model suffix
    train_features_135m.columns = [f"{col}_135m" for col in train_features_135m.columns]
    train_features_360m.columns = [f"{col}_360m" for col in train_features_360m.columns]

    # Combine training features
    X_train = pd.concat([train_features_135m, train_features_360m], axis=1)
    y_train = train_135m['label']

    # Prepare validation features
    val_features_135m = val_135m[feature_cols].copy()
    val_features_360m = val_360m[feature_cols].copy()

    # Add model suffix
    val_features_135m.columns = [f"{col}_135m" for col in val_features_135m.columns]
    val_features_360m.columns = [f"{col}_360m" for col in val_features_360m.columns]

    # Combine validation features
    X_val = pd.concat([val_features_135m, val_features_360m], axis=1)
    y_val = val_135m['label']

    # Handle infinite/NaN values
    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(X_train.median())
    X_val = X_val.replace([np.inf, -np.inf], np.nan).fillna(X_train.median())

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    print(f"Training set: {X_train.shape[0]} samples")
    print(f"Validation set: {X_val.shape[0]} samples")
    print(f"Features: {X_train.shape[1]}")

    # Train classifiers
    classifiers = {
        'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
    }

    results = {}

    for name, clf in classifiers.items():
        print(f"\nTraining {name}...")

        if name == 'Logistic Regression':
            clf.fit(X_train_scaled, y_train)
            y_pred_proba = clf.predict_proba(X_val_scaled)[:, 1]
        else:
            clf.fit(X_train, y_train)
            y_pred_proba = clf.predict_proba(X_val)[:, 1]

        # Calculate metrics
        auc = roc_auc_score(y_val, y_pred_proba)

        # Calculate TPR@FPR=0.1
        fpr, tpr, thresholds = roc_curve(y_val, y_pred_proba)
        target_fpr = 0.1
        idx = np.argmin(np.abs(fpr - target_fpr))
        tpr_at_fpr_01 = tpr[idx]
        actual_fpr = fpr[idx]

        # Binary predictions
        y_pred = (y_pred_proba > 0.5).astype(int)
        accuracy = (y_pred == y_val).mean()

        print(f"AUC: {auc:.4f}")
        print(f"TPR@FPR=0.1: {tpr_at_fpr_01:.4f} (actual FPR: {actual_fpr:.4f})")
        print(f"Accuracy: {accuracy:.4f}")

        results[name] = {
            'classifier': clf,
            'scaler': scaler if name == 'Logistic Regression' else None,
            'auc': auc,
            'tpr_at_fpr_01': tpr_at_fpr_01,
            'accuracy': accuracy,
            'feature_names': X_train.columns.tolist()
        }

        # Feature importance for Random Forest
        if name == 'Random Forest':
            importance_df = pd.DataFrame({
                'feature': X_train.columns,
                'importance': clf.feature_importances_
            }).sort_values('importance', ascending=False)

            print("\nTop 10 Most Important Features:")
            for idx, row in importance_df.head(10).iterrows():
                print(f"  {row['feature']}: {row['importance']:.4f}")

    return results

# Train the classifier
classifier_results = build_membership_classifier(
    train_results_135m, train_results_360m,
    val_results_135m, val_results_360m
)

Building membership inference classifier...
Training set: 50000 samples
Validation set: 10000 samples
Features: 18

Training Logistic Regression...
AUC: 0.8852
TPR@FPR=0.1: 0.6492 (actual FPR: 0.1000)
Accuracy: 0.8119

Training Random Forest...
AUC: 0.8842
TPR@FPR=0.1: 0.6554 (actual FPR: 0.0976)
Accuracy: 0.8050

Top 10 Most Important Features:
  loss_360m: 0.1258
  perplexity_360m: 0.1252
  min_token_prob_360m: 0.0780
  mean_entropy_135m: 0.0729
  perplexity_135m: 0.0717
  loss_135m: 0.0708
  low_conf_ratio_360m: 0.0640
  mean_entropy_360m: 0.0607
  low_conf_ratio_135m: 0.0520
  std_token_prob_360m: 0.0517


## Results Summary

Final performance metrics for the membership inference attack.

In [11]:
# Results summary
print("MEMBERSHIP INFERENCE ATTACK RESULTS")
print("=" * 50)

for name, results in classifier_results.items():
    print(f"\n{name}:")
    print(f"  AUC: {results['auc']:.4f}")
    print(f"  TPR@FPR=0.1: {results['tpr_at_fpr_01']:.4f}")
    print(f"  Accuracy: {results['accuracy']:.4f}")

# Best model
best_model = max(classifier_results.values(), key=lambda x: x['auc'])
print(f"\nBEST MODEL PERFORMANCE:")
print(f"AUC: {best_model['auc']:.4f}")
print(f"TPR@FPR=0.1: {best_model['tpr_at_fpr_01']:.4f}")
print(f"Accuracy: {best_model['accuracy']:.4f}")

print(f"\nDataset processed:")
print(f"Training samples: {len(train_results_135m)}")
print(f"Validation samples: {len(val_results_135m)}")
print(f"Total features: {len(best_model['feature_names'])}")

MEMBERSHIP INFERENCE ATTACK RESULTS

Logistic Regression:
  AUC: 0.8852
  TPR@FPR=0.1: 0.6492
  Accuracy: 0.8119

Random Forest:
  AUC: 0.8842
  TPR@FPR=0.1: 0.6554
  Accuracy: 0.8050

BEST MODEL PERFORMANCE:
AUC: 0.8852
TPR@FPR=0.1: 0.6492
Accuracy: 0.8119

Dataset processed:
Training samples: 50000
Validation samples: 10000
Total features: 18


## Save Results

Save the trained models and results for future use.

In [12]:
# Save results and models
import pickle

# Save the best classifier
best_classifier_name = max(classifier_results.keys(), key=lambda x: classifier_results[x]['auc'])
best_classifier = classifier_results[best_classifier_name]

# Save to files
with open('membership_classifier.pkl', 'wb') as f:
    pickle.dump(best_classifier, f)

# Save feature results
train_results_135m.to_csv('train_features_135m.csv', index=False)
train_results_360m.to_csv('train_features_360m.csv', index=False)
val_results_135m.to_csv('val_features_135m.csv', index=False)
val_results_360m.to_csv('val_features_360m.csv', index=False)

print("Results saved:")
print("- membership_classifier.pkl")
print("- train_features_135m.csv")
print("- train_features_360m.csv")
print("- val_features_135m.csv")
print("- val_features_360m.csv")

Results saved:
- membership_classifier.pkl
- train_features_135m.csv
- train_features_360m.csv
- val_features_135m.csv
- val_features_360m.csv


## Generate Test Predictions for Kaggle Submission

Use the best classifier to generate predictions on the test set and create a submission file.

In [13]:
def prepare_test_predictions(test_135m, test_360m, best_classifier):
    """
    Prepare test features and generate predictions for Kaggle submission.
    """
    print("Preparing test features for prediction...")

    # Feature columns
    feature_cols = ['perplexity', 'loss', 'mean_token_prob', 'min_token_prob', 'max_token_prob',
                   'std_token_prob', 'mean_entropy', 'top_k_mass', 'low_conf_ratio']

    # Extract features
    test_features_135m = test_135m[feature_cols].copy()
    test_features_360m = test_360m[feature_cols].copy()

    # Add model suffix
    test_features_135m.columns = [f"{col}_135m" for col in test_features_135m.columns]
    test_features_360m.columns = [f"{col}_360m" for col in test_features_360m.columns]

    # Combine features
    X_test = pd.concat([test_features_135m, test_features_360m], axis=1)

    # Handle infinite/NaN values (use same strategy as training)
    X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(X_test.median())

    # Apply scaling if the classifier uses it
    if best_classifier['scaler'] is not None:
        X_test = best_classifier['scaler'].transform(X_test)

    # Generate predictions
    print("Generating predictions on test set...")
    test_predictions = best_classifier['classifier'].predict_proba(X_test)[:, 1]

    return test_predictions

# Get best classifier
best_classifier_name = max(classifier_results.keys(), key=lambda x: classifier_results[x]['auc'])
best_classifier = classifier_results[best_classifier_name]

print(f"Using {best_classifier_name} for test predictions (AUC: {best_classifier['auc']:.4f})")

# Generate test predictions
test_predictions = prepare_test_predictions(test_results_135m, test_results_360m, best_classifier)

print(f"Generated {len(test_predictions)} predictions")
print(f"Prediction range: [{test_predictions.min():.4f}, {test_predictions.max():.4f}]")
print(f"Mean prediction: {test_predictions.mean():.4f}")

Using Logistic Regression for test predictions (AUC: 0.8852)
Preparing test features for prediction...
Generating predictions on test set...
Generated 15000 predictions
Prediction range: [0.0001, 1.0000]
Mean prediction: 0.5003


In [ ]:
# Create Kaggle submission file with correct IDs
print("Creating Kaggle submission file...")

# Use the  'id' column from test dataset
test_ids = test_df['id'].tolist()

# Create submission DataFrame with proper ID mapping
submission_df = pd.DataFrame({
    'id': test_ids,
    'is_member': test_predictions
})

# Save submission file
submission_filename = 'membership_inference_submission.csv'
submission_df.to_csv(submission_filename, index=False)

print(f"Kaggle submission saved: {submission_filename}")
print(f"Shape: {submission_df.shape}")
print(f"ID range: {submission_df['id'].min()} to {submission_df['id'].max()}")
print(f"Prediction range: [{submission_df['is_member'].min():.4f}, {submission_df['is_member'].max():.4f}]")

print("\nFirst 5 rows:")
print(submission_df.head())


Creating Kaggle submission file...
Kaggle submission saved: membership_inference_submission.csv
Shape: (15000, 2)
ID range: 130000 to 37499
Prediction range: [0.0001, 1.0000]

First 5 rows:
       id  is_member
0   32656   0.860779
1  137254   0.218944
2  134884   0.601773
3   33065   0.561611
4   34304   0.829173

READY FOR KAGGLE! Upload: membership_inference_submission.csv
